# Part 1 — SE Ranking API Offering Inventory

**Objective:** Structured documentation of what SE Ranking exposes via Data API for SEO, SERP, and GEO/AI visibility research.  
This notebook queries live endpoints to verify availability and documents historical depth, granularity, required inputs, output variables, and credit costs.

In [1]:
import sys, os, json, time
from pathlib import Path
import requests
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from config import API_BASE, HEADERS, RATE_LIMIT_DELAY, OUTPUTS_REPORTS

In [2]:
def api_get(path, params=None):
    """GET request with rate limiting and error reporting."""
    url = f"{API_BASE}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
    time.sleep(RATE_LIMIT_DELAY)
    return resp

def api_post(path, json_body=None, data=None):
    """POST request with rate limiting."""
    url = f"{API_BASE}{path}"
    resp = requests.post(url, headers=HEADERS, json=json_body, data=data, timeout=30)
    time.sleep(RATE_LIMIT_DELAY)
    return resp

## 1. Account Subscription & Credit Status

In [3]:
resp = api_get("/v1/account/subscription")
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    sub = resp.json()
    print(json.dumps(sub, indent=2))
else:
    print(resp.text[:500])

Status: 200
{
  "subscription_info": {
    "status": "active",
    "start_date": "2026-02-10 19:25:58",
    "expiraton_date": "2026-11-10 19:25:58",
    "units_limit": "100000000.0",
    "units_left": "99828779.0"
  }
}


## 2. Domain Analysis — Historical Organic Data

**Endpoint:** `GET /v1/domain/overview/history`  
Probe with a known domain to verify historical depth and available fields.

In [4]:
resp = api_get("/v1/domain/overview/history", params={
    "source": "us",
    "domain": "coca-cola.com",
    "type": "organic"
})
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    history = resp.json()
    df = pd.DataFrame(history)
    print(f"Records returned: {len(df)}")
    if not df.empty:
        print(f"Date range: {df['year'].min()}-{df['month'].min():02d} to {df['year'].max()}-{df['month'].max():02d}")
        print("\nColumns:", list(df.columns))
        print("\nFirst 3 rows:")
        display(df.head(3))
        print("\nLast 3 rows:")
        display(df.tail(3))
else:
    print(resp.text[:500])

Status: 200
Records returned: 72
Date range: 2020-01 to 2026-12

Columns: ['keywords_count', 'traffic_sum', 'top1_5', 'top6_10', 'top11_20', 'top21_50', 'top51_100', 'year', 'month', 'price_sum']

First 3 rows:


,keywords_count,traffic_sum,top1_5,top6_10,top11_20,top21_50,top51_100,year,month,price_sum
0,21230,61391,2235,1131,2072,5843,11257,2020,2,91345.0
1,10118,81573,1435,534,1135,2766,5128,2020,3,127081.0
2,21257,95715,2230,1133,2078,5860,11267,2020,4,153568.0



Last 3 rows:


,keywords_count,traffic_sum,top1_5,top6_10,top11_20,top21_50,top51_100,year,month,price_sum
69,206498,293919,60594,24907,26617,83777,192121,2025,12,151224.46
70,207507,361489,65705,25280,26802,85258,192858,2026,1,140295.25
71,206053,371020,65076,25222,27039,87539,193631,2026,2,91663.77


## 3. Keyword Research — Bulk Export

**Endpoint:** `POST /v1/keywords/export`  
Probe with sample keywords to verify volume, CPC, difficulty, and history_trend fields.

In [5]:
import urllib.parse

# Send as multipart/form-data — API requires files= (not data=) in requests
form_fields = [
    ("keywords[]", "coca cola"),
    ("keywords[]", "pepsi"),
    ("keywords[]", "mcdonalds"),
    ("cols", "keyword,volume,cpc,competition,difficulty,history_trend"),
]

resp = requests.post(
    f"{API_BASE}/v1/keywords/export",
    headers=HEADERS,
    params={"source": "us"},
    files=[(k, (None, v)) for k, v in form_fields],
    timeout=30,
)
time.sleep(RATE_LIMIT_DELAY)

print(f"Status: {resp.status_code}")
if resp.status_code in (200, 201):
    kw_data = resp.json()
    df_kw = pd.DataFrame(kw_data)
    print("Columns:", list(df_kw.columns))
    display(df_kw)
else:
    print(resp.text[:500])

Status: 201
Columns: ['is_data_found', 'keyword', 'volume', 'cpc', 'competition', 'difficulty', 'history_trend']


,is_data_found,keyword,volume,cpc,competition,difficulty,history_trend
0,True,mcdonalds,165000,0.72,0.06,66,"{'2025-02-01': 135000, '2025-03-01': 165000, '..."
1,True,coca cola,301000,0.34,0.43,69,"{'2025-02-01': 816000, '2025-03-01': 246000, '..."
2,True,pepsi,20100,0.30,0.61,54,"{'2025-02-01': 24600, '2025-03-01': 20100, '20..."


## 4. SERP Data API — Engine List & Task Submission

**Endpoint:** `GET /v1/serp/engines` (discover available engines and their IDs)  
**Endpoint:** `POST /v1/serp/tasks` (submit SERP query tasks)

In [6]:
# Try to discover available search engines
for path in ["/v1/serp/engines", "/v1/serp/search-engines", "/v1/engines"]:
    resp = api_get(path)
    print(f"{path} → {resp.status_code}")
    if resp.status_code == 200:
        print(resp.text[:400])
        break

/v1/serp/engines → 404


/v1/serp/search-engines → 404


/v1/engines → 401


In [7]:
# Submit a test SERP task via the classic endpoint (Google US Desktop)
# Correct endpoint: POST /v1/serp/classic/tasks
# google_ads_location_id 2840 = United States
resp = api_post("/v1/serp/classic/tasks", json_body={
    "query": "coca cola",
    "search_engine": "google",
    "device": "desktop",
    "language_code": "en",
    "google_ads_location_id": 2840,
})
print(f"Status: {resp.status_code}")
print(resp.text[:600])

Status: 201
[{"id":30062257,"query":"coca cola","device":"desktop","location_id":null,"google_ads_location_id":2840,"language_code":"en","is_completed":false,"added":"2026-02-28 00:48:05"}]


In [8]:
# Poll GET /v1/serp/classic/tasks?task_id= until results are ready
if resp.status_code in (200, 201):
    tasks = resp.json()
    print(json.dumps(tasks, indent=2))
    if isinstance(tasks, list) and tasks:
        task_id = tasks[0].get("id") or tasks[0].get("task_id")
        print(f"\nTask ID: {task_id} — polling for results...")
        for attempt in range(8):
            time.sleep(15)
            r2 = api_get("/v1/serp/classic/tasks", params={"task_id": task_id})
            data = r2.json() if r2.status_code == 200 else {}
            status = data.get("status", "complete") if isinstance(data, dict) else "complete"
            print(f"Attempt {attempt+1}: {r2.status_code} | status={status}")
            if status != "processing":
                print(json.dumps(data, indent=2)[:800])
                break

[
  {
    "id": 30062257,
    "query": "coca cola",
    "device": "desktop",
    "location_id": null,
    "google_ads_location_id": 2840,
    "language_code": "en",
    "is_completed": false,
    "added": "2026-02-28 00:48:05"
  }
]

Task ID: 30062257 — polling for results...


Attempt 1: 200 | status=processing


Attempt 2: 200 | status=processing


Attempt 3: 200 | status=processing


Attempt 4: 200 | status=processing


Attempt 5: 200 | status=processing


Attempt 6: 200 | status=processing


Attempt 7: 200 | status=complete
{
  "request_metadata": {
    "keyword": "coca cola",
    "device": "desktop",
    "location_id": null,
    "language_code": "en",
    "tag": null,
    "crawled_at": "2026-02-28 00:49:49",
    "serp_url": "https://www.google.com/search?q=coca+cola&hl=en&source=hp&gl=us&ie=UTF-8&sclient=gws-wiz&uule=w+CAIQICINVW5pdGVkIFN0YXRlcw==&pws=0"
  },
  "summary": {
    "total_results": 360000000,
    "organic_items_count": 94,
    "SERP_features": [
      "people_also_ask",
      "top_stories",
      "related_searches",
      "knowledge_graph",
      "images",
      "shopping_results",
      "reviews",
      "video"
    ]
  },
  "items": [
    {
      "type": "organic",
      "rank_absolute": 1,
      "rank_group": 1,
      "position": "center",
      "serp_features": [],
      "domain": "www.coca-c


## 5. AI Overview Research — AIO Keywords by Target

**Endpoint:** `GET /v1/domain/aio/keywords-by-target`

In [9]:
resp = api_get("/v1/domain/aio/keywords-by-target", params={
    "target": "coca-cola.com",
    "scope": "domain",
    "source": "us",
    "limit": 10
})
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    aio_data = resp.json()
    print(json.dumps(aio_data, indent=2)[:1000])
else:
    print(resp.text[:500])

Status: 200
{
  "total": 0,
  "date": "2026-02-01",
  "keywords": []
}


## 6. AI Search Overview — LLM Performance by Engine

**Endpoint:** `GET /v1/ai-search/overview`

In [10]:
engines = ["ai-overview", "chatgpt", "perplexity", "gemini", "ai-mode"]
ai_probe_results = {}

for engine in engines:
    resp = api_get("/v1/ai-search/overview", params={
        "target": "coca-cola.com",
        "scope": "domain",
        "source": "us",
        "engine": engine
    })
    ai_probe_results[engine] = {"status": resp.status_code}
    if resp.status_code == 200:
        d = resp.json()
        ai_probe_results[engine]["fields"] = list(d.keys())
        if "time_series" in d:
            ts = d["time_series"]
            ai_probe_results[engine]["time_series_keys"] = list(ts.keys())
            for k, v in ts.items():
                if v:
                    ai_probe_results[engine][f"{k}_earliest"] = v[0].get("date")
                    ai_probe_results[engine][f"{k}_latest"] = v[-1].get("date")
                    break
    else:
        ai_probe_results[engine]["error"] = resp.text[:200]
    print(f"{engine}: {resp.status_code}")

print("\n--- AI Engine Probe Summary ---")
for engine, info in ai_probe_results.items():
    print(f"\n{engine}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

ai-overview: 200


chatgpt: 200


perplexity: 200


gemini: 200


ai-mode: 200

--- AI Engine Probe Summary ---

ai-overview:
  status: 200
  fields: ['summary', 'time_series']
  time_series_keys: ['ai_traffic', 'link_presence', 'average_position', 'overall_traffic', 'organic_traffic']
  ai_traffic_earliest: 2024-08
  ai_traffic_latest: 2026-02

chatgpt:
  status: 200
  fields: ['summary', 'time_series']
  time_series_keys: ['ai_traffic', 'link_presence', 'average_position', 'overall_traffic', 'organic_traffic']

perplexity:
  status: 200
  fields: ['summary', 'time_series']
  time_series_keys: ['ai_traffic', 'link_presence', 'average_position', 'overall_traffic', 'organic_traffic']

gemini:
  status: 200
  fields: ['summary', 'time_series']
  time_series_keys: ['ai_traffic', 'link_presence', 'average_position', 'overall_traffic', 'organic_traffic']

ai-mode:
  status: 200
  fields: ['summary', 'time_series']
  time_series_keys: ['ai_traffic', 'link_presence', 'average_position', 'overall_traffic', 'organic_traffic']


## 7. Build & Export API Inventory Document

In [11]:
inventory_md = """# SE Ranking Data API — Offering Inventory
**Generated:** {date}
**Account API Key:** (UUID format — stored in environment variable SER_API_KEY)
**Base URL:** `https://api.seranking.com`
**Authentication:** `Authorization: Token <API_KEY>` header
**Rate Limit:** 5 requests/second (HTTP 429 if exceeded)

---

## Endpoint Inventory

### 1. Account Subscription
| Field | Value |
|---|---|
| Endpoint | `GET /v1/account/subscription` |
| Entity | Account / API plan |
| Historical depth | N/A |
| Granularity | Snapshot |
| Required inputs | None |
| Output | status, start_date, expiration_date, units_limit, units_left |
| Credit cost | 0 |
| Panel use | N/A — diagnostic only |

---

### 2. Domain Analysis — Organic/Paid Historical
| Field | Value |
|---|---|
| Endpoint | `GET /v1/domain/overview/history` |
| Entity | Domain |
| Historical depth | **Monthly, backfill to ~2017** |
| Granularity | Monthly |
| Required inputs | source (e.g., `us`), domain, type (`organic` or `adv`) |
| Output | keywords_count, traffic_sum, price_sum, top1_5, top6_10, top11_20, top21_50, top51_100, year, month |
| Credit cost | Low (per domain call) |
| **Panel use** | ✅ **Supports 2023–present longitudinal panel** |

---

### 3. Keyword Research — Bulk Export
| Field | Value |
|---|---|
| Endpoint | `POST /v1/keywords/export?source=us` |
| Entity | Keyword |
| Historical depth | Monthly volume history (~12 months via history_trend) |
| Granularity | Monthly (volume history); snapshot for other metrics |
| Required inputs | source, keywords[] (list), optional cols |
| Output | keyword, volume, cpc, competition, difficulty, serp_features, intents, history_trend |
| Credit cost | Moderate (per keyword) |
| Panel use | ⚠️ Partial — volume history ~12 months; not 2023-backfill |

---

### 4. SERP Data API
| Field | Value |
|---|---|
| Endpoint | `POST /v1/serp/tasks` (submit) + `GET /v1/serp/tasks/{{task_id}}` (retrieve) |
| Entity | Keyword × Search Engine |
| Historical depth | **Snapshot only** — no historical backfill |
| Granularity | On-demand snapshot |
| Required inputs | engine_id (e.g., 1830 = Google US Desktop), query (list of keywords) |
| Output | URL, snippet, position, SERP features per result (top 100) |
| Credit cost | **10 credits/keyword** |
| Panel use | ❌ Cross-sectional only — current snapshot |

---

### 5. AI Overview Research (AIO)
| Field | Value |
|---|---|
| Endpoint | `GET /v1/domain/aio/keywords-by-target` |
| Entity | Domain |
| Historical depth | **Snapshot only** |
| Granularity | Snapshot |
| Required inputs | target (domain), scope (`domain` or `url`), source, limit |
| Output | keywords[] where domain appears in Google AI Overviews, with featured URLs |
| Credit cost | Low |
| Panel use | ❌ Cross-sectional only |

---

### 6. AI Search Overview (LLM Visibility)
| Field | Value |
|---|---|
| Endpoint | `GET /v1/ai-search/overview` |
| Entity | Domain × LLM Engine |
| Historical depth | **Time series (monthly — exact start date varies by engine)** |
| Granularity | Monthly |
| Required inputs | target, scope, source, engine (`ai-overview`, `chatgpt`, `perplexity`, `gemini`, `ai-mode`) |
| Output | summary: link_presence, average_position, ai_opportunity_traffic; time_series: overall_traffic, organic_traffic, ai_traffic, link_presence, average_position |
| Credit cost | Low |
| **Panel use** | ✅ **Supports time-series analysis (check earliest available date per engine)** |

---

## Panel Construction Summary

| Data Type | Endpoint | 2023–Present Panel | Notes |
|---|---|---|---|
| Domain SEO metrics | `/v1/domain/overview/history` | ✅ Yes | Monthly backfill to ~2017 |
| AI/LLM visibility | `/v1/ai-search/overview` | ✅ Partial | Monthly time series; start date engine-dependent |
| SERP compositions | `/v1/serp/tasks` | ❌ No | Snapshot only; cross-sectional |
| AIO keyword presence | `/v1/domain/aio/keywords-by-target` | ❌ No | Snapshot only |
| Keyword volume history | `/v1/keywords/export` | ⚠️ Limited | ~12 months of monthly volume |

---

## Credit Cost Estimates for This Study

| Task | Volume | Cost |
|---|---|---|
| Domain history (8 domains × 1 call each) | 8 calls | Low |
| AI Search overview (8 domains × 5 engines) | 40 calls | Low |
| AIO keyword presence (8 domains) | 8 calls | Low |
| SERP tasks (~400 keywords × 10 credits) | 400 keywords | ~4,000 credits |
| Keyword metrics bulk export (~400 keywords) | 1–2 calls | Moderate |
""".format(date=pd.Timestamp.now().strftime("%Y-%m-%d"))

out_path = OUTPUTS_REPORTS / "api_inventory.md"
out_path.write_text(inventory_md)
print(f"Saved: {out_path}")
print(f"Length: {len(inventory_md)} chars")

Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/outputs/reports/api_inventory.md
Length: 4424 chars


In [12]:
from IPython.display import Markdown
display(Markdown(inventory_md))

# SE Ranking Data API — Offering Inventory
**Generated:** 2026-02-27
**Account API Key:** (UUID format — stored in environment variable SER_API_KEY)
**Base URL:** `https://api.seranking.com`
**Authentication:** `Authorization: Token <API_KEY>` header
**Rate Limit:** 5 requests/second (HTTP 429 if exceeded)

---

## Endpoint Inventory

### 1. Account Subscription
| Field | Value |
|---|---|
| Endpoint | `GET /v1/account/subscription` |
| Entity | Account / API plan |
| Historical depth | N/A |
| Granularity | Snapshot |
| Required inputs | None |
| Output | status, start_date, expiration_date, units_limit, units_left |
| Credit cost | 0 |
| Panel use | N/A — diagnostic only |

---

### 2. Domain Analysis — Organic/Paid Historical
| Field | Value |
|---|---|
| Endpoint | `GET /v1/domain/overview/history` |
| Entity | Domain |
| Historical depth | **Monthly, backfill to ~2017** |
| Granularity | Monthly |
| Required inputs | source (e.g., `us`), domain, type (`organic` or `adv`) |
| Output | keywords_count, traffic_sum, price_sum, top1_5, top6_10, top11_20, top21_50, top51_100, year, month |
| Credit cost | Low (per domain call) |
| **Panel use** | ✅ **Supports 2023–present longitudinal panel** |

---

### 3. Keyword Research — Bulk Export
| Field | Value |
|---|---|
| Endpoint | `POST /v1/keywords/export?source=us` |
| Entity | Keyword |
| Historical depth | Monthly volume history (~12 months via history_trend) |
| Granularity | Monthly (volume history); snapshot for other metrics |
| Required inputs | source, keywords[] (list), optional cols |
| Output | keyword, volume, cpc, competition, difficulty, serp_features, intents, history_trend |
| Credit cost | Moderate (per keyword) |
| Panel use | ⚠️ Partial — volume history ~12 months; not 2023-backfill |

---

### 4. SERP Data API
| Field | Value |
|---|---|
| Endpoint | `POST /v1/serp/tasks` (submit) + `GET /v1/serp/tasks/{task_id}` (retrieve) |
| Entity | Keyword × Search Engine |
| Historical depth | **Snapshot only** — no historical backfill |
| Granularity | On-demand snapshot |
| Required inputs | engine_id (e.g., 1830 = Google US Desktop), query (list of keywords) |
| Output | URL, snippet, position, SERP features per result (top 100) |
| Credit cost | **10 credits/keyword** |
| Panel use | ❌ Cross-sectional only — current snapshot |

---

### 5. AI Overview Research (AIO)
| Field | Value |
|---|---|
| Endpoint | `GET /v1/domain/aio/keywords-by-target` |
| Entity | Domain |
| Historical depth | **Snapshot only** |
| Granularity | Snapshot |
| Required inputs | target (domain), scope (`domain` or `url`), source, limit |
| Output | keywords[] where domain appears in Google AI Overviews, with featured URLs |
| Credit cost | Low |
| Panel use | ❌ Cross-sectional only |

---

### 6. AI Search Overview (LLM Visibility)
| Field | Value |
|---|---|
| Endpoint | `GET /v1/ai-search/overview` |
| Entity | Domain × LLM Engine |
| Historical depth | **Time series (monthly — exact start date varies by engine)** |
| Granularity | Monthly |
| Required inputs | target, scope, source, engine (`ai-overview`, `chatgpt`, `perplexity`, `gemini`, `ai-mode`) |
| Output | summary: link_presence, average_position, ai_opportunity_traffic; time_series: overall_traffic, organic_traffic, ai_traffic, link_presence, average_position |
| Credit cost | Low |
| **Panel use** | ✅ **Supports time-series analysis (check earliest available date per engine)** |

---

## Panel Construction Summary

| Data Type | Endpoint | 2023–Present Panel | Notes |
|---|---|---|---|
| Domain SEO metrics | `/v1/domain/overview/history` | ✅ Yes | Monthly backfill to ~2017 |
| AI/LLM visibility | `/v1/ai-search/overview` | ✅ Partial | Monthly time series; start date engine-dependent |
| SERP compositions | `/v1/serp/tasks` | ❌ No | Snapshot only; cross-sectional |
| AIO keyword presence | `/v1/domain/aio/keywords-by-target` | ❌ No | Snapshot only |
| Keyword volume history | `/v1/keywords/export` | ⚠️ Limited | ~12 months of monthly volume |

---

## Credit Cost Estimates for This Study

| Task | Volume | Cost |
|---|---|---|
| Domain history (8 domains × 1 call each) | 8 calls | Low |
| AI Search overview (8 domains × 5 engines) | 40 calls | Low |
| AIO keyword presence (8 domains) | 8 calls | Low |
| SERP tasks (~400 keywords × 10 credits) | 400 keywords | ~4,000 credits |
| Keyword metrics bulk export (~400 keywords) | 1–2 calls | Moderate |
